In [ ]:
import requests
import json
import math

### 새로운 버전
# 1. API 기본 정보 설정
base_url = "http://apis.data.go.kr/1230000/ao/PubDataOpnStdService"
operation_name = "getDataSetOpnStdBidPblancInfo"
url = f"{base_url}/{operation_name}"

# 2. 요청 메시지(파라미터) 설정
# ServiceKey는 공공데이터포털에서 발급받은 인증키를 사용해야 합니다 (필수 항목).
# 아래 'YOUR_SERVICE_KEY' 부분을 실제 발급받은 키로 반드시 교체해주세요.
service_key = "qPYIYI7lUydoOKfx4lfTj1Bddm3sLaMp0/sbhMz/b9aWP89T9aZZaBsIzd9yapSE/5fyXGO8Q2mpdzBlEGYj+Q==" 

# 한 번에 가져올 데이터 수를 최대로 설정하여 API 호출 횟수를 줄이는 것이 효율적입니다.
# (API가 허용하는 최대치가 있다면 그 값을 사용하는 것이 좋습니다. 여기서는 100으로 설정)
rows_per_page = 999

params = {
    'numOfRows': str(rows_per_page),  # 한 페이지 결과 수 [3, 4]
    'pageNo': '1',      # 페이지 번호 [3, 4]
    'ServiceKey': service_key,
    'type': 'json',     
    'bidNtceBgnDt': '202509180000', 
    'bidNtceEndDt': '202509192359'  # 검색 기간을 늘려 더 많은 데이터를 테스트
}

search_keyword = "세무"
# 3. API 호출 및 전체 데이터 수집
all_results = []
try:
    # --- 1단계: 첫 페이지 호출 및 전체 페이지 수 계산 ---
    print("첫 번째 페이지 데이터를 요청합니다...")
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    if data['response']['header']['resultCode'] != '00':
        raise Exception(f"API 오류: {data['response']['header']['resultMsg']}")

    # 전체 결과 수(totalCount)와 한 페이지 결과 수(numOfRows) 가져오기 [1, 2]
    total_count = int(data['response']['body']['totalCount'])
    num_of_rows = int(data['response']['body']['numOfRows'])

    if total_count == 0:
        print("해당 기간에 조회된 입찰 공고 데이터가 없습니다.")
    else:
        # 첫 페이지 결과 저장
        if 'items' in data['response']['body'] and data['response']['body']['items']:
            all_results.extend(data['response']['body']['items'])
        
        # 전체 페이지 수 계산
        total_pages = math.ceil(total_count / num_of_rows)
        print(f"전체 데이터: {total_count}건, 전체 페이지: {total_pages}페이지")

        # --- 2단계: 나머지 페이지 순회하며 데이터 수집 ---
        for page in range(2, total_pages + 1):
            params['pageNo'] = str(page)
            print(f"{page}/{total_pages} 페이지 데이터를 요청합니다...")
            
            response = requests.get(url, params=params)
            response.raise_for_status()
            data = response.json()

            if data['response']['header']['resultCode'] == '00' and 'items' in data['response']['body']:
                all_results.extend(data['response']['body']['items'])
            else:
                print(f"{page} 페이지 로딩 중 문제 발생: {data['response']['header']['resultMsg']}")

        # --- 3단계: 수집된 전체 데이터에 대해 필터링 수행 ---
        print("\n--- 전체 데이터 수집 완료. 필터링을 시작합니다. ---")
        filtered_results = [
            item for item in all_results 
            if search_keyword in item.get('bidNtceNm', '')
        ]

        # 4. 필터링된 최종 결과 출력
        print(f"--- 총 {len(all_results)}개 공고 중 '{search_keyword}' 키워드가 포함된 공고 ({len(filtered_results)}개) ---")
        if filtered_results:
            print(json.dumps(filtered_results, indent=4, ensure_ascii=False))
        else:
            print("키워드를 포함한 공고가 없습니다.")

except requests.exceptions.RequestException as e:
    print(f"API 호출 중 오류가 발생했습니다: {e}")
except json.JSONDecodeError:
    print("JSON 형식으로 변환할 수 없습니다. 응답 내용:")
    print(response.text)
except Exception as e:
    print(f"처리 중 오류 발생: {e}")


In [ ]:
import requests
import json
import math

# 1. API 기본 정보 설정
base_url = "http://apis.data.go.kr/1230000/ao/PubDataOpnStdService"
operation_name = "getDataSetOpnStdBidPblancInfo"
url = f"{base_url}/{operation_name}"

# 2. 요청 메시지(파라미터) 설정
# ServiceKey는 공공데이터포털에서 발급받은 인증키를 사용해야 합니다 (필수 항목).
# 아래 'YOUR_SERVICE_KEY' 부분을 실제 발급받은 키로 반드시 교체해주세요.
service_key = "qPYIYI7lUydoOKfx4lfTj1Bddm3sLaMp0/sbhMz/b9aWP89T9aZZaBsIzd9yapSE/5fyXGO8Q2mpdzBlEGYj+Q==" 

# 한 번에 가져올 데이터 수를 최대로 설정하여 API 호출 횟수를 줄이는 것이 효율적입니다.
# (API가 허용하는 최대치가 있다면 그 값을 사용하는 것이 좋습니다. 여기서는 100으로 설정)
rows_per_page = 999

params = {
    'numOfRows': str(rows_per_page),  # 한 페이지 결과 수 [3, 4]
    'pageNo': '1',      # 페이지 번호 [3, 4]
    'ServiceKey': service_key,
    'type': 'json',     
    'bidNtceBgnDt': '202509180000', 
    'bidNtceEndDt': '202509192359'  # 검색 기간을 늘려 더 많은 데이터를 테스트
}

search_keyword = "세무"
# 3. API 호출 및 전체 데이터 수집
all_results = []
try:
    # --- 1단계: 첫 페이지 호출 및 전체 페이지 수 계산 ---
    print("첫 번째 페이지 데이터를 요청합니다...")
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    if data['response']['header']['resultCode'] != '00':
        raise Exception(f"API 오류: {data['response']['header']['resultMsg']}")

    # 전체 결과 수(totalCount)와 한 페이지 결과 수(numOfRows) 가져오기 [1, 2]
    total_count = int(data['response']['body']['totalCount'])
    num_of_rows = int(data['response']['body']['numOfRows'])

    if total_count == 0:
        print("해당 기간에 조회된 입찰 공고 데이터가 없습니다.")
    else:
        # 첫 페이지 결과 저장
        if 'items' in data['response']['body'] and data['response']['body']['items']:
            all_results.extend(data['response']['body']['items'])
        
        # 전체 페이지 수 계산
        total_pages = math.ceil(total_count / num_of_rows)
        print(f"전체 데이터: {total_count}건, 전체 페이지: {total_pages}페이지")

        # --- 2단계: 나머지 페이지 순회하며 데이터 수집 ---
        for page in range(2, total_pages + 1):
            params['pageNo'] = str(page)
            print(f"{page}/{total_pages} 페이지 데이터를 요청합니다...")
            
            response = requests.get(url, params=params)
            response.raise_for_status()
            data = response.json()

            if data['response']['header']['resultCode'] == '00' and 'items' in data['response']['body']:
                all_results.extend(data['response']['body']['items'])
            else:
                print(f"{page} 페이지 로딩 중 문제 발생: {data['response']['header']['resultMsg']}")

        # --- 3단계: 수집된 전체 데이터에 대해 필터링 수행 ---
        print("\n--- 전체 데이터 수집 완료. 필터링을 시작합니다. ---")
        filtered_results = [
            item for item in all_results 
            if search_keyword in item.get('bidNtceNm', '')
        ]

        # 4. 필터링된 최종 결과 출력
        print(f"--- 총 {len(all_results)}개 공고 중 '{search_keyword}' 키워드가 포함된 공고 ({len(filtered_results)}개) ---")
        if filtered_results:
            print(json.dumps(filtered_results, indent=4, ensure_ascii=False))
        else:
            print("키워드를 포함한 공고가 없습니다.")

except requests.exceptions.RequestException as e:
    print(f"API 호출 중 오류가 발생했습니다: {e}")
except json.JSONDecodeError:
    print("JSON 형식으로 변환할 수 없습니다. 응답 내용:")
    print(response.text)
except Exception as e:
    print(f"처리 중 오류 발생: {e}")


In [57]:
# 3. API 호출 및 전체 데이터 수집
all_results = []
try:
    # --- 1단계: 첫 페이지 호출 및 전체 페이지 수 계산 ---
    print("첫 번째 페이지 데이터를 요청합니다...")
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    if data['response']['header']['resultCode'] != '00':
        raise Exception(f"API 오류: {data['response']['header']['resultMsg']}")

    # 전체 결과 수(totalCount)와 한 페이지 결과 수(numOfRows) 가져오기 [1, 2]
    total_count = int(data['response']['body']['totalCount'])
    num_of_rows = int(data['response']['body']['numOfRows'])

    if total_count == 0:
        print("해당 기간에 조회된 입찰 공고 데이터가 없습니다.")
    else:
        # 첫 페이지 결과 저장
        if 'items' in data['response']['body'] and data['response']['body']['items']:
            all_results.extend(data['response']['body']['items'])
        
        # 전체 페이지 수 계산
        total_pages = math.ceil(total_count / num_of_rows)
        print(f"전체 데이터: {total_count}건, 전체 페이지: {total_pages}페이지")

        # --- 2단계: 나머지 페이지 순회하며 데이터 수집 ---
        for page in range(2, total_pages + 1):
            params['pageNo'] = str(page)
            print(f"{page}/{total_pages} 페이지 데이터를 요청합니다...")
            
            response = requests.get(url, params=params)
            response.raise_for_status()
            data = response.json()

            if data['response']['header']['resultCode'] == '00' and 'items' in data['response']['body']:
                all_results.extend(data['response']['body']['items'])
            else:
                print(f"{page} 페이지 로딩 중 문제 발생: {data['response']['header']['resultMsg']}")

        # --- 3단계: 수집된 전체 데이터에 대해 필터링 수행 ---
        print("\n--- 전체 데이터 수집 완료. 필터링을 시작합니다. ---")
        filtered_results = [
            item for item in all_results 
            if search_keyword in item.get('bidNtceNm', '')
        ]

        # 4. 필터링된 최종 결과 출력
        print(f"--- 총 {len(all_results)}개 공고 중 '{search_keyword}' 키워드가 포함된 공고 ({len(filtered_results)}개) ---")
        if filtered_results:
            print(json.dumps(filtered_results, indent=4, ensure_ascii=False))
        else:
            print("키워드를 포함한 공고가 없습니다.")

except requests.exceptions.RequestException as e:
    print(f"API 호출 중 오류가 발생했습니다: {e}")
except json.JSONDecodeError:
    print("JSON 형식으로 변환할 수 없습니다. 응답 내용:")
    print(response.text)
except Exception as e:
    print(f"처리 중 오류 발생: {e}")

첫 번째 페이지 데이터를 요청합니다...
전체 데이터: 3033건, 전체 페이지: 4페이지
2/4 페이지 데이터를 요청합니다...
3/4 페이지 데이터를 요청합니다...
4/4 페이지 데이터를 요청합니다...

--- 전체 데이터 수집 완료. 필터링을 시작합니다. ---
--- 총 3130개 공고 중 '세무' 키워드가 포함된 공고 (1개) ---
[
    {
        "bidNtceNo": "R25BK01061960",
        "bidNtceOrd": "000",
        "refNtceNo": "R25BK01061960",
        "refNtceOrd": "000",
        "ppsNtceYn": "Y",
        "bidNtceNm": "강동세무서 청사 신축공사 건설폐기물 처리용역",
        "bidNtceSttusNm": "일반공고",
        "bidNtceDate": "2025-09-18",
        "bidNtceBgn": "15:09",
        "bsnsDivNm": "용역",
        "intrntnlBidYn": "N",
        "cmmnCntrctYn": "Y",
        "cmmnReciptMethdNm": "분담이행",
        "elctrnBidYn": "Y",
        "cntrctCnclsSttusNm": "총액계약",
        "cntrctCnclsMthdNm": "수의계약",
        "bidwinrDcsnMthdNm": "",
        "ntceInsttNm": "조달청 서울지방조달청",
        "ntceInsttCd": "1230121",
        "ntceInsttOfclDeptNm": "",
        "ntceInsttOfclNm": "채윤정",
        "ntceInsttOfclTel": "070-405-68795",
        "ntceInsttOfclEmailAdrs": "yjung6

In [33]:
try:
    response = requests.get(url, params=params)
    response.raise_for_status()  # HTTP 오류 발생 시 예외 발생

    # 4. 응답 메시지 처리 및 출력
    print("--- API 응답 메시지 ---")
    
    # 응답이 JSON 형식일 경우, 예쁘게 출력
    if params['type'] == 'json':
        try:
            data = response.json()
            # ensure_ascii=False 옵션으로 한글이 깨지지 않게 합니다.
            print(json.dumps(data, indent=4, ensure_ascii=False))
        except json.JSONDecodeError:
            print("JSON 형식으로 변환할 수 없습니다. 응답 내용:")
            print(response.text)
    else: # XML 또는 기타 형식의 응답
        print(response.text)

except requests.exceptions.RequestException as e:
    print(f"API 호출 중 오류가 발생했습니다: {e}")

--- API 응답 메시지 ---
{
    "response": {
        "header": {
            "resultCode": "00",
            "resultMsg": "정상"
        },
        "body": {
            "items": [
                {
                    "bidNtceNo": "R25BK00933743",
                    "bidNtceOrd": "000",
                    "refNtceNo": "R25BK00933743",
                    "refNtceOrd": "000",
                    "ppsNtceYn": "Y",
                    "bidNtceNm": "2025년 경기미 가공저장시설 스마트화 지원사업 현미 색채선별기 구매(긴급)",
                    "bidNtceSttusNm": "일반공고",
                    "bidNtceDate": "2025-07-01",
                    "bidNtceBgn": "07:49",
                    "bsnsDivNm": "물품",
                    "intrntnlBidYn": "N",
                    "cmmnCntrctYn": "Y",
                    "cmmnReciptMethdNm": "",
                    "elctrnBidYn": "Y",
                    "cntrctCnclsSttusNm": "총액계약",
                    "cntrctCnclsMthdNm": "제한경쟁",
                    "bidwinrDcsnMthdNm": "적격심사제",
                

In [29]:
# 3. API 호출 및 데이터 필터링

search_keyword = "이전가격"

try:
    response = requests.get(url, params=params)
    response.raise_for_status()  # HTTP 오류 발생 시 예외 발생

    data = response.json()

    # API 응답 성공 여부 확인 (resultCode가 '00'이면 정상)
    if data['response']['header']['resultCode'] == '00':
        
        # 'items'가 있는지, 비어있지 않은지 확인
        if 'items' in data['response']['body'] and data['response']['body']['items']:
            all_items = data['response']['body']['items']
            
            # --- 여기서 필터링이 수행됩니다 ---
            # bidNtceNm에 search_keyword가 포함된 item만 추출합니다.
            filtered_results = [
                item for item in all_items 
                if search_keyword in item.get('bidNtceNm', '')
            ]

            # 4. 필터링된 결과 출력
            print(f"--- 총 {len(all_items)}개 공고 중 '{search_keyword}' 키워드가 포함된 공고 ({len(filtered_results)}개) ---")
            if filtered_results:
                # ensure_ascii=False 옵션으로 한글이 깨지지 않게 합니다.
                print(json.dumps(filtered_results, indent=4, ensure_ascii=False))
            else:
                print("키워드를 포함한 공고가 없습니다.")
                print(json.dumps(data, indent=4, ensure_ascii=False))
        
        else:
            print("해당 기간에 조회된 입찰 공고 데이터가 없습니다.")

    else:
        # API 에러 메시지 출력
        print(f"API 오류: {data['response']['header']['resultMsg']}")


except requests.exceptions.RequestException as e:
    print(f"API 호출 중 오류가 발생했습니다: {e}")
except json.JSONDecodeError:
    print("JSON 형식으로 변환할 수 없습니다. 응답 내용:")
    print(response.text)
except KeyError:
    print("응답 데이터의 구조가 예상과 다릅니다. 전체 응답을 확인하세요.")
    print(response.text)


--- 총 10개 공고 중 '이전가격' 키워드가 포함된 공고 (0개) ---
키워드를 포함한 공고가 없습니다.
{
    "response": {
        "header": {
            "resultCode": "00",
            "resultMsg": "정상"
        },
        "body": {
            "items": [
                {
                    "bidNtceNo": "R25BK00823203",
                    "bidNtceOrd": "000",
                    "refNtceNo": "R25BK00823203",
                    "refNtceOrd": "000",
                    "ppsNtceYn": "Y",
                    "bidNtceNm": "북안면 용계리 상수관로 부설공사 관급자재(PFP강관 및 소켓)",
                    "bidNtceSttusNm": "일반공고",
                    "bidNtceDate": "2025-05-01",
                    "bidNtceBgn": "04:57",
                    "bsnsDivNm": "물품",
                    "intrntnlBidYn": "N",
                    "cmmnCntrctYn": "Y",
                    "cmmnReciptMethdNm": "",
                    "elctrnBidYn": "Y",
                    "cntrctCnclsSttusNm": "총액계약",
                    "cntrctCnclsMthdNm": "수의계약",
                    "bidwinrDcs